# Section 1: Why Traditional Patterns Break
## AI-Native Software Architecture | O'Reilly Course

We start with the simplest possible LLM architecture:

**One prompt → one model call → one output**

This is how many LLM applications begin. Run it repeatedly to see why generating an answer is not the same as producing a reliable system output.

In [ ]:
import os
import support_utils.llm_client as llm_client

from support_utils import call_llm, primary_issue

# False: credential-free dummy LLM
# True: Vertex AI when configured, otherwise OpenAI
USE_REAL_LLM = False
llm_client.USE_REAL_LLM = USE_REAL_LLM

if not USE_REAL_LLM:
    provider = "Dummy LLM"
elif llm_client.gemini_client is not None:
    provider = f"Vertex AI ({llm_client.GEMINI_MODEL})"
elif os.getenv("OPENAI_API_KEY"):
    provider = f"OpenAI ({llm_client.OPENAI_MODEL})"
else:
    provider = "No real LLM configured"

print(f"Provider: {provider}")
print(f"Customer issue: {primary_issue}")

## Hands-On: Why Naive LLM Apps Break

### Task

Run the same naive LLM flow multiple times using the same customer issue.

### Observe

- Does the wording or structure vary?
- Does the model make different assumptions?
- Does the recommended action change?
- Does the response claim authority the system does not have?
- Could another service reliably consume the response?

### Expected outcome

See why a single prompt and model call does not provide a reliable production contract.

In [ ]:
def naive_support_assistant(issue: str) -> str:
    prompt = f"""Help the customer with this issue:

{issue}
"""
    return call_llm(prompt, temperature=0.9)


naive_outputs = [
    naive_support_assistant(primary_issue)
    for _ in range(5)
]

print("=== Naive outputs across repeated runs ===")

for run_number, output in enumerate(naive_outputs, start=1):
    print(f"\nRun {run_number}:")
    print(output)

## What Did We Observe?

The input stayed the same, but the responses may differ in:

- wording and structure
- assumptions about the customer’s issue
- recommended next action
- claims about refund eligibility or authorization
- downstream usability

The model produced a response, but the application did not define or enforce what a valid response should be.

## Failure Modes Exposed by the Naive Pattern

The naive prompt only asks the model to help. It does not define:

- the required output structure
- which evidence to use
- what actions the model may recommend or perform
- which requests are in scope
- how sensitive information should be handled

Not every run will demonstrate every failure, but the current design provides no protection against any of them.

In [ ]:
failure_modes = [
    {
        "failure_mode": "Unstructured output",
        "why_it_matters": "Downstream services cannot reliably parse or validate the response.",
        "example": "A free-form paragraph instead of a defined structure.",
    },
    {
        "failure_mode": "Unauthorized action",
        "why_it_matters": "The model may claim it performed an action the system cannot authorize.",
        "example": "Your refund has been processed.",
    },
    {
        "failure_mode": "Ungrounded claims",
        "why_it_matters": "The model may invent policies or facts without supporting evidence.",
        "example": "Refunds are always available.",
    },
    {
        "failure_mode": "Scope drift",
        "why_it_matters": "The model may respond to requests outside the application’s intended domain.",
        "example": "Explaining linked lists in a billing assistant.",
    },
    {
        "failure_mode": "Sensitive-data mishandling",
        "why_it_matters": "The model may request, repeat, or expose information that should be protected.",
        "example": "Repeating a customer’s SSN in the response.",
    },
]

for failure in failure_modes:
    print(f"\n🔴 {failure['failure_mode']}")
    print(f"Why it matters: {failure['why_it_matters']}")
    print(f"Example: {failure['example']}")

## Section 1 Takeaway

The naive pattern is useful for prototyping, but it provides no enforceable guarantees about:

- output structure
- consistency
- grounding
- permitted actions
- downstream usability

The model generates a candidate response. The surrounding architecture must determine whether that response is usable.

**Next:** Section 2: controlling behavior through input and output contracts.